In [48]:
import pandas as pd
from pathlib import Path
import preprocess as pp
from summary_utils.y_variable_analysis import analyze_y_variables

In [49]:
def load_data(file_path):
    return pd.read_parquet(file_path)
Path('output_data').mkdir(exist_ok=True)
Path('output_data/plots').mkdir(exist_ok=True)
Path('output_data/stats').mkdir(exist_ok=True)

df = load_data('/home/azureuser/FactorLab_earnings_estimates/input_data/universe_with_affactor.parquet')

In [50]:
df.columns[:40]

Index(['Unnamed: 0', 'companyid', 'fiscalyear', 'fiscalquarter', 'EPS_actual',
       'EPS_actual_et', 'EPSDiff', 'EPS_surprise', 'EPS_count', 'EPS_std',
       'EPS_guidance_high', 'EPS_guidance_low', 'EPSNormalized_actual',
       'EPSNormalized_actual_et', 'EPSNormalized_diff',
       'EPSNormalized_surprise', 'EPSNormalized_count', 'EPSNormalized_std',
       'EPSNormalized_guidance_high', 'EPSNormalized_guidance_low',
       'revenue_actual', 'revenue_actual_et', 'revenueDiff',
       'revenue_surprise', 'revenue_count', 'revenue_std',
       'revenue_guidance_high', 'revenue_guidance_low', 'affactor_asofdate_0',
       '60MAVGTTMEP', '60MAVGTTMROA', '60MAVGTTMROE', 'AdjAstAdjChg3YFCF',
       'AdjIntCov', 'AstAdjChg3YFCF', 'BVEV', 'BVEV_2', 'BuyBackChg',
       'CurLiaP', 'DIVIDENDGROWTH'],
      dtype='object')

In [51]:
# sum the number of numeric variables
num_numeric_vars = df.select_dtypes(include='number').shape[1]
print(f"Number of numeric variables: {num_numeric_vars}")

Number of numeric variables: 576


In [52]:
df['year'] = pd.to_datetime(df['EPS_actual_et']).dt.year
# check rows with year < 2012 from columns start from 20
df_before_2012 = df[df['year'] < 2012].iloc[:, 20:]
print(df_before_2012['BVEV'].notna().sum()/ df_before_2012.shape[0])

# check rows with year >= 2012 from columns start from 20
df_after_2012 = df[df['year'] >= 2012].iloc[:, 20:]
print(df_after_2012['BVEV'].notna().sum() / df_after_2012.shape[0])

df_after_2012

0.0765841376416282


0.2692632935349427


,revenue_actual,revenue_actual_et,revenueDiff,revenue_surprise,revenue_count,revenue_std,revenue_guidance_high,revenue_guidance_low,affactor_asofdate_0,60MAVGTTMEP,...,OEP,ProvChgOffSales,QuickRatio,RskAdjRS,SP,SalesAcc,SolvencyRatio,StdErr180D,WCTurn,year
28596,158.27300,2012-01-03 16:01:00-05:00,4.26,2.7661,6.0,6.04773,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012
28597,136.34000,2012-01-03 16:30:00-05:00,2.75,2.0585,6.0,2.30505,134.0,130.0,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012
28598,313.02500,2012-01-04 09:27:05-05:00,16.55,5.5822,7.0,3.49595,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012
28599,128.27900,2012-01-04 16:01:00-05:00,-1.29,-0.9956,15.0,1.59663,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012
28600,3014.50000,2012-01-04 16:18:00-05:00,-186.38,-5.8228,12.0,129.81244,NaN,NaN,2011-12-31,0.068,...,0.097,NaN,2.703,0.394,0.481,-2.068,0.826,-20.176,2.456,2012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127925,104.57500,2024-04-01 17:00:00-04:00,NaN,NaN,3.0,1.41611,100.0,90.0,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024
127926,28.36895,2024-04-15 16:41:35-04:00,NaN,NaN,4.0,1.02324,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024
127927,3.53900,2024-04-16 17:19:41-04:00,NaN,NaN,1.0,0.00000,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024
127928,2288.00000,2024-02-27 19:00:00-05:00,NaN,NaN,3.0,8.77212,NaN,NaN,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024


In [55]:
df = pd.read_parquet('/home/azureuser/FactorLab_earnings_estimates/output_data/cleaned_data.parquet')
df.columns[:40]

Index(['companyid', 'y_EPS_actual', 'y_EPSDiff', 'y_EPS_surprise',
       'x_EPS_count', 'x_EPS_std', 'x_EPS_guidance_high', 'x_EPS_guidance_low',
       'y_EPSNormalized_actual', 'y_EPSNormalized_diff',
       'y_EPSNormalized_surprise', 'x_EPSNormalized_count',
       'x_EPSNormalized_std', 'x_EPSNormalized_guidance_high',
       'x_EPSNormalized_guidance_low', 'y_revenue_actual', 'y_revenueDiff',
       'y_revenue_surprise', 'x_revenue_count', 'x_revenue_std',
       'x_revenue_guidance_high', 'x_revenue_guidance_low', 'x_60MAVGTTMEP',
       'x_60MAVGTTMROA', 'x_60MAVGTTMROE', 'x_AstAdjChg3YFCF', 'x_EstDiffC',
       'x_FinLev', 'x_IO_TO', 'x_LTGC', 'x_NIStab', 'x_OCFEqt', 'x_OCFStab',
       'x_PAYOUTRATIO', 'x_ROASTAB', 'x_ROEStab', 'x_SEV', 'x_LogUnadjPrice',
       'x_stdRR36M', 'x_AssetTurn_2'],
      dtype='object')